In [1]:
import sys
from pathlib import Path

# Add ../Retrieval to Python path
RETRIEVAL_PATH = Path("..") / "Retrieval"
sys.path.append(str(RETRIEVAL_PATH.resolve()))


In [2]:
import sys, os, csv
import torch, torchaudio
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple
from dotenv import load_dotenv
from Classes.TextAdaptationModule import TextAdaptationModule
from Classes.DataRetrieval import DataRetrieval
from Classes.OpenAIClient import OpenAIClient
from Classes.GeminiClient import GeminiClient
from Classes.InstructionAnalysisModule import InstructionAnalysisModule
from IPython.display import Audio

import numpy as np
import soundfile as sf

sys.path.append(os.path.abspath("../Retrieval"))
sys.path.append(os.path.abspath("../CosyVoice"))
sys.path.append(os.path.abspath("../CosyVoice/third_party/Matcha-TTS"))

# CosyVoice imports
try:
    from modelscope import snapshot_download
    from cosyvoice.cli.cosyvoice import CosyVoice2
    from cosyvoice.utils.file_utils import load_wav
    
    model_path = snapshot_download("iic/CosyVoice2-0.5B")
    
except Exception as e:
    raise RuntimeError("Could not import CosyVoice2 / load_wav. Fix your CosyVoice install or imports.") from e

c:\Users\Admin\miniconda3\envs\cosyvoice\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


failed to import ttsfrd, use wetext instead


2025-08-28 00:28:12,216 DEBUG Starting new HTTPS connection (1): www.modelscope.cn:443


2025-08-28 00:28:13,627 DEBUG https://www.modelscope.cn:443 "GET /api/v1/models/iic/CosyVoice2-0.5B/revisions HTTP/1.1" 200 None
2025-08-28 00:28:14,290 DEBUG https://www.modelscope.cn:443 "GET /api/v1/models/iic/CosyVoice2-0.5B/repo/files?Revision=master&Recursive=True HTTP/1.1" 200 None
2025-08-28 00:28:14,295 - modelscope - INFO - Creating symbolic link C:\Users\Admin\.cache\modelscope\hub\iic\iic/CosyVoice2-0___5B -> C:\Users\Admin\.cache\modelscope\hub\iic/CosyVoice2-0.5B.
2025-08-28 00:28:14,296 - modelscope - WARNING - Failed to create symbolic link C:\Users\Admin\.cache\modelscope\hub\iic\iic/CosyVoice2-0___5B -> C:\Users\Admin\.cache\modelscope\hub\iic/CosyVoice2-0.5B: [WinError 1314] A required privilege is not held by the client: 'C:\\Users\\Admin\\.cache\\modelscope\\hub\\iic\\iic\\CosyVoice2-0___5B' -> 'C:\\Users\\Admin\\.cache\\modelscope\\hub\\iic/CosyVoice2-0.5B'


In [3]:
def ensure_dir(p: Path) -> None:
    p.mkdir(parents=True, exist_ok=True)

def to_float32(audio: Any) -> Tuple[np.ndarray, int]:
    if isinstance(audio, tuple) and len(audio) == 2:
        arr, sr = audio
    elif isinstance(audio, dict) and "audio" in audio and "sample_rate" in audio:
        arr, sr = audio["audio"], audio["sample_rate"]
    else:
        arr, sr = audio, 16000

    arr = np.asarray(arr)
    try:
        arr = arr.detach().cpu().numpy()
    except Exception:
        pass

    if arr.dtype != np.float32:
        arr = arr.astype(np.float32)

    if arr.ndim > 1:
        if arr.shape[0] < arr.shape[-1]:
            arr = arr.T
        arr = arr.mean(axis=1)
    return arr, int(sr)

def _peak_normalize_float32(arr: np.ndarray, peak_db: float = -3.0) -> np.ndarray:
    if arr is None or arr.size == 0:
        return arr
    peak = float(np.max(np.abs(arr)))
    if peak <= 0.0:
        return arr
    target = 10 ** (peak_db / 20.0)
    return arr * (target / peak)

def save_audio_anything(chunks, out_path, default_sr=24000):
    """
    Save CosyVoice outputs into one WAV at the correct sample rate.

    Detect audio keys: audio/wav/pcm/tts_speech/speech/samples
    Detect SR keys: sample_rate/sr/tts_sr/sampleRate/sample_rate_hz/sampling_rate/rate/fs/hz
    Falls back to default_sr (pass DEFAULT_TTS_SR from the model).
    """
    if not chunks:
        return None, "No chunks returned from TTS call"

    KEY_CANDIDATES = ["audio", "wav", "pcm", "tts_speech", "speech", "samples"]
    SR_CANDIDATES  = ["sample_rate", "sr", "tts_sr", "sampleRate",
                      "sample_rate_hz", "sampling_rate", "rate", "fs", "hz"]

    audios = []
    detected_sr = None

    for ch in chunks:
        arr, sr = None, None

        if isinstance(ch, dict):
            for k in KEY_CANDIDATES:
                if k in ch and ch[k] is not None:
                    arr = ch[k]
                    break
            for k in SR_CANDIDATES:
                if k in ch and ch[k] is not None:
                    try:
                        sr = int(ch[k])
                    except Exception:
                        pass
                    break

        if arr is None:
            if isinstance(ch, tuple) and len(ch) == 2:
                arr, sr = ch  # (audio, sr)
            else:
                arr = ch

        try:
            a, s = to_float32({"audio": arr, "sample_rate": sr if sr else default_sr})
            if a is None or a.ndim == 0 or a.size == 0:
                continue
            audios.append(a)
            if sr:
                detected_sr = s
        except Exception:
            continue

    if not audios:
        return None, "No valid audio arrays found in chunks"

    final_sr = int(detected_sr if detected_sr else default_sr)
    concat = np.concatenate(audios, axis=0)
    concat = _peak_normalize_float32(concat, peak_db=-3.0)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    sf.write(str(out_path), concat, final_sr)
    return out_path, None


In [4]:
print("[INFO] Loading CosyVoice model...")
cosyvoice = CosyVoice2(model_path, load_jit=False, load_trt=False, load_vllm=False, fp16=False)
print("[INFO] CosyVoice ready.")

DEFAULT_TTS_SR = (
    getattr(cosyvoice, "tts_sample_rate", None)
    or getattr(cosyvoice, "sample_rate", None)
    or 24000   # fallback
)
print("DEFAULT_TTS_SR =", DEFAULT_TTS_SR)

# Load API key
load_dotenv()
openai_api_key = os.getenv("OPENAI_API_KEY")
if not openai_api_key:
    raise ValueError("OPENAI_API_KEY not found in .env")

gemini_api_key = os.getenv("GEMINI_API_KEY")
if not gemini_api_key:
    raise ValueError("GEMINI_API_KEY not found in .env")

[INFO] Loading CosyVoice model...


c:\Users\Admin\miniconda3\envs\cosyvoice\lib\site-packages\diffusers\models\lora.py:393: FutureWarning: `LoRACompatibleLinear` is deprecated and will be removed in version 1.0.0. Use of `LoRACompatibleLinear` is deprecated. Please switch to PEFT backend by installing PEFT: `pip install peft`.
  deprecate("LoRACompatibleLinear", "1.0.0", deprecation_message)
2025-08-28 00:28:19,300 INFO input frame rate=25
c:\Users\Admin\miniconda3\envs\cosyvoice\lib\site-packages\torch\nn\utils\weight_norm.py:28: UserWarning: torch.nn.utils.weight_norm is deprecated in favor of torch.nn.utils.parametrizations.weight_norm.
  warnings.warn("torch.nn.utils.weight_norm is deprecated in favor of torch.nn.utils.parametrizations.weight_norm.")
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
c:\Users\Admin\miniconda3\envs\cos

2025-08-28 00:28:23,370 DEBUG https://www.modelscope.cn:443 "GET /api/v1/models/pengzhendong/wetext/revisions HTTP/1.1" 200 205
2025-08-28 00:28:23,796 DEBUG https://www.modelscope.cn:443 "GET /api/v1/models/pengzhendong/wetext/repo/files?Revision=master&Recursive=True HTTP/1.1" 200 None
2025-08-28 00:28:23,955 DEBUG Starting new HTTPS connection (1): www.modelscope.cn:443


2025-08-28 00:28:25,894 DEBUG https://www.modelscope.cn:443 "GET /api/v1/models/pengzhendong/wetext/revisions HTTP/1.1" 200 205
2025-08-28 00:28:26,308 DEBUG https://www.modelscope.cn:443 "GET /api/v1/models/pengzhendong/wetext/repo/files?Revision=master&Recursive=True HTTP/1.1" 200 None


[INFO] CosyVoice ready.
DEFAULT_TTS_SR = 24000


In [ ]:


import json
from pathlib import Path

INPUT_JSON = Path("./2_adapted_text_generation/1_prompts_with_speaker_info_noimplicit_final.json")


OUT_DIR = Path("CosyVoice_Baseline")
# ensure_dir(OUT_DIR)

if not INPUT_JSON.exists():
    raise FileNotFoundError(f"JSON not found at {INPUT_JSON}. Please set INPUT_JSON to your file path.")

with open(INPUT_JSON, "r", encoding="utf-8") as f:
    data = json.load(f)

rows = []
scenario_stats = []   # will store dicts: {"scen_idx": int, "num_instructions": int, "standard_sentence": str}
skipped_count = 0




for scen_idx, scen in enumerate(data, start=1):
    # Top-level fields per scenario
    standard_sentence = (scen.get("standard_sentence") or "").strip()

    if len(standard_sentence) >= 2 and standard_sentence[0] == standard_sentence[-1] and standard_sentence[0] in ("'", '"'):
        standard_sentence_clean = standard_sentence[1:-1]
    else:
        standard_sentence_clean = standard_sentence

    valid_instr = 0
    
    adaptations = scen.get("adaptations") or []
    for adapt_idx, adapt in enumerate(adaptations, start=1):
        instr = (adapt.get("explicit_instruction") or "").strip()
        # Skip if missing or placeholder error
        if not standard_sentence_clean or not instr or instr.startswith("[ERROR]"):
            continue

        accent = (adapt.get("gt_accent")
                  or adapt.get("results", {}).get("inferred_speaker_info", {}).get("accent")
                  or "UNK")
        gender = (adapt.get("gt_gender")
                  or adapt.get("results", {}).get("inferred_speaker_info", {}).get("gender")
                  or "U")
        age = (adapt.get("gt_age")
               or adapt.get("results", {}).get("inferred_speaker_info", {}).get("age")
               or "NA")

        def norm(x):
            return str(x).replace(" ", "").replace("[", "").replace("]", "").replace(",", "-").replace("/", "_")

        uid = f"sc{scen_idx:03d}_a{norm(accent)}_g{norm(gender)}_age{norm(age)}_{adapt_idx:02d}"

        rows.append({
            "uid": uid,
            "text": standard_sentence_clean,        # standard sentence to speak
            "instruction": instr                    # explicit instruction
        })
        valid_instr += 1

    scenario_stats.append({
        "scen_idx": scen_idx,
        "num_instructions": valid_instr,
        "standard_sentence": standard_sentence_clean
        })

print(f"[INFO] Loaded {len(rows)} rows from {INPUT_JSON}")

num_scenarios = len(scenario_stats)
total_valid_instructions = sum(s["num_instructions"] for s in scenario_stats)
total_wavs_to_save = len(rows)

print(f"[INFO] Loaded {num_scenarios} scenarios from {INPUT_JSON}")
print(f"[COUNT] Total valid instructions: {total_valid_instructions}")
print(f"[COUNT] Total WAVs to be saved:  {total_wavs_to_save}")
if skipped_count:
    print(f"[COUNT] Skipped rows (missing text/instruction or error placeholders): {skipped_count}")


[INFO] Loaded 3600 rows from 2_adapted_text_generation\1_prompts_with_speaker_info_noimplicit_final.json
[INFO] Loaded 30 scenarios from 2_adapted_text_generation\1_prompts_with_speaker_info_noimplicit_final.json
[COUNT] Total valid instructions: 3600
[COUNT] Total WAVs to be saved:  3600


In [ ]:
import os
import torchaudio
import numpy as np
import soundfile as sf
from pathlib import Path
from IPython.display import Audio, display

# config
INPUT_SAMPLE_RATE = 16000
OUTPUT_SAMPLE_RATE = 24000
NEAR_SILENT_PATH = "./assets/_near_silence_2s_16k.wav"
NEAR_SILENT_TENSOR = load_wav(NEAR_SILENT_PATH, INPUT_SAMPLE_RATE)

instr_module = InstructionAnalysisModule(api_key=gemini_api_key) 
adapt_module = TextAdaptationModule(api_key=openai_api_key)


import os, traceback
from IPython.display import Audio, display


import os, csv, traceback
from IPython.display import Audio, display

MANIFEST_PATH = OUT_DIR / "instruct2_manifest.tsv"
fieldnames = ["uid", "standard_sentence", "explicit_instruction", "output_filename", "output_path", "error"]


def _prep_near_silence(t, sr=16000, max_len_s=0.5, target_rms=1e-6):
    import torch
    if not isinstance(t, torch.Tensor):
        try:
            import numpy as np
            t = torch.from_numpy(np.asarray(t))
        except Exception:
            pass
    if t.ndim == 1: t = t.unsqueeze(0)
    if t.size(0) > 1: t = t.mean(dim=0, keepdim=True)
    # trim + set tiny RMS
    max_n = int(max_len_s * sr)
    if t.shape[-1] > max_n: t = t[..., :max_n]
    cur = t.pow(2).mean().sqrt().item() if t.numel() else 0.0
    if cur > 0:
        t = t * (target_rms / cur)
    else:
        t = torch.randn_like(t) * target_rms
    return t.contiguous().float()

NEAR_SILENT_TENSOR = _prep_near_silence(NEAR_SILENT_TENSOR, sr=INPUT_SAMPLE_RATE)
MODEL_OUT_SR = int(getattr(cosyvoice, "sample_rate", 24000))

# Fallback collector
def _collect_instruct2_chunks(out_obj, debug_label=""):
    """
    Normalize CosyVoice inference_instruct2 outputs into a list the saver can handle.
    Accepts:
      - single torch.Tensor
      - dict with tts_speech/audio/wav/pcm
      - (audio, sr) tuples
      - generators/iterables yielding any of the above
    """
    chunks = []

    def _accept(x):
        if isinstance(x, dict):
            if "tts_speech" in x and "tts_sr" not in x and "tts_sps" in x:
                x["tts_sr"] = x["tts_sps"]
            chunks.append(x)
            return
        
        if isinstance(x, (tuple, list)) and len(x) >= 1:
            if len(x) >= 2 and isinstance(x[1], (int, np.integer)):
                chunks.append((x[0], int(x[1])))
            else:
                chunks.append(x[0])
            return
        
        if torch.is_tensor(x) or isinstance(x, np.ndarray):
            chunks.append(x)
            return

    if torch.is_tensor(out_obj) or isinstance(out_obj, dict) or isinstance(out_obj, np.ndarray):
        _accept(out_obj)
    else:
        items = None
        try:
            items = list(out_obj)
        except TypeError:
            _accept(out_obj)

        if items is not None:
            if not items:
                print(f"[DBG] inference_instruct2 yielded 0 items ({debug_label}). type={type(out_obj)}")
            for it in items:
                _accept(it)

    return chunks

with open(MANIFEST_PATH, "w", encoding="utf-8", newline="") as mf:
    writer = csv.DictWriter(mf, delimiter="\t", fieldnames=fieldnames)
    writer.writeheader()

    for i, sample in enumerate(rows, start=1):
        uid = sample["uid"]
        explicit_instruction = sample["instruction"]
        standard_sentence = sample["text"]
        out_path = OUT_DIR / f"{i:04d}_{uid}.wav"

        print(f"\n=== {uid} ===")
        print("Explicit Instruction:", explicit_instruction)
        print("Standard Sentence   :", standard_sentence)

        error_msg = ""
        saved_path_str = ""
        try:
            # Use near-silent tensor as the prompt
            prompt_tensor = NEAR_SILENT_TENSOR

            spk_id = f"__noref__{uid}"
            cosyvoice.add_zero_shot_spk("", prompt_tensor, spk_id)

            # Call inference_instruct2(text, instruction, prompt_speech_16k, zero_shot_spk_id)
            out_iter = cosyvoice.inference_instruct2(
                standard_sentence,
                explicit_instruction,
                prompt_tensor,
            )

            # Collect chunks robustly for different CosyVoice return shapes
            chunks = _collect_instruct2_chunks(out_iter)
            if not chunks:
                raise RuntimeError("No audio chunks returned from inference_instruct2.")

            saved_path, err = save_audio_anything(chunks, out_path, default_sr=OUTPUT_SAMPLE_RATE)
            if err:
                raise RuntimeError(err)

            saved_path_str = str(saved_path)
            print(f"[SAVED] -> {saved_path_str}")
            # display(Audio(filename=saved_path_str))

        except Exception as e:
            error_msg = repr(e)
            print(f"[ERROR] UID: {uid} —", error_msg)
            traceback.print_exc()

        # Write a manifest row for each attempt (success or error)
        writer.writerow({
            "uid": uid,
            "standard_sentence": standard_sentence,
            "explicit_instruction": explicit_instruction,
            "output_filename": out_path.name if saved_path_str else "",
            "output_path": saved_path_str,
            "error": error_msg,
        })

print(f"\n[MANIFEST] Wrote: {MANIFEST_PATH}")


In [ ]:
# --- Sanity snapshot: planned vs saved vs failed ---

from pathlib import Path
import csv, json

INPUT_JSON = Path("./2_adapted_text_generation/1_prompts_with_speaker_info_noimplicit_final.json")
MANIFEST_PATH = Path("./CosyVoice_Baseline/CosyVoice_Baseline.tsv")

# How many scenarios, target adaptations, and how many are valid (non-[ERROR])?
with open(INPUT_JSON, "r", encoding="utf-8") as f:
    data = json.load(f)

num_scenarios = len(data)
target_total = sum(len(s.get("adaptations") or []) for s in data)

# Count invalids
invalids = []
valid_total = 0
for scen_idx, scen in enumerate(data, start=1):
    txt = (scen.get("standard_sentence") or "").strip()
    if len(txt) >= 2 and txt[0] == txt[-1] and txt[0] in ("'", '"'):
        txt = txt[1:-1]
    for adapt_idx, adapt in enumerate(scen.get("adaptations") or [], start=1):
        instr = (adapt.get("explicit_instruction") or "").strip()
        if not txt or not instr or instr.startswith("[ERROR]"):
            invalids.append((scen_idx, adapt_idx, instr))
        else:
            valid_total += 1

print(f"[SCENARIOS] {num_scenarios}")
print(f"[ADAPTATIONS in JSON] {target_total}")
print(f"[INVALID skipped rows] {len(invalids)} (expected 3)")
print(f"[VALID rows your loop runs] {valid_total}")

# What actually got saved? (using manifest + filesystem)
saved = 0
failed = 0
missing_on_disk = 0

if MANIFEST_PATH.exists():
    with open(MANIFEST_PATH, "r", encoding="utf-8", newline="") as f:
        r = csv.DictReader(f, delimiter="\t")
        rows = list(r)

    for row in rows:
        err = (row.get("error") or "").strip()
        outp = (row.get("output_path") or "").strip()
        if err:
            failed += 1
        elif outp:
            if Path(outp).exists():
                saved += 1
            else:
                missing_on_disk += 1

    print(f"[MANIFEST] rows: {len(rows)}")
    print(f"[SAVED files] {saved}")
    print(f"[FAILED (error set)] {failed}")
    print(f"[MISSING on disk despite path in manifest] {missing_on_disk}")
else:
    print(f"[WARN] Manifest not found at {MANIFEST_PATH}")

# show first few invalids
# for k,(si,ai,instr) in enumerate(invalids[:5], start=1):
#     print(f"  - Invalid #{k}: scenario {si}, adaptation {ai}, instr={repr(instr[:60])}")

[SCENARIOS] 30
[ADAPTATIONS in JSON] 3600
[INVALID skipped rows] 0 (expected 3)
[VALID rows your loop runs] 3600
[WARN] Manifest not found at CosyVoice_Baseline\CosyVoice_Baseline.tsv
